In [3]:
%pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.3 MB 6.3 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/9.3 MB 7.8 MB/s eta 0:00:01
   --------------- ------------------------ 3.7/9.3 MB 6.3 MB/s eta 0:00:01
   -------------------- ------------------- 4.7/9.3 MB 6.1 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.3 MB 5.6 MB/s eta 0:00:01
   -------------------------- ------------- 6.3/9.3 MB 5.2 MB/s eta 0:00:01
   ----------------------------- ---------- 6.8/9.3 MB 4.9 MB/s eta 0:00:01
   -------------------------------- ------- 7.6/9.3 MB 4.6 MB/s eta 0:00:01
   ---------------------------------- ---

In [2]:
"""
Career Launch (CL26) — Pre-Program Survey: Descriptive Stats & Charts
======================================================================

Purpose
-------
Summarizes how students felt about their internship match and the broader
placement process, BEFORE the internship started. Produces:
  1. A set of PNG charts (saved to ./charts/)
  2. A summary_tables.json file with all the numbers used in the charts
     and in the Word report (so the Word-doc build step doesn't need to
     touch pandas at all).

Input
-----
Pre_Program_Survey_Data-Career_Launch.csv  (1,749 student responses)

Key assumptions / methodology notes
------------------------------------
- Likert-style items are mapped to numeric scores so we can report a mean
  alongside the response distribution:
    * 5-point agreement/satisfaction scales -> 1 (lowest) to 5 (highest)
    * "Satisfaction Rating" column is already provided on a 0-100 scale by
      the survey tool (Very dissatisfied=0 ... Very satisfied=100); we use
      it directly for the match-satisfaction headline number.
    * 3-point helpfulness/clarity scales -> 1 (lowest) to 3 (highest)
  Responses like "I did not attend", "I don't remember", or blank are
  treated as non-substantive and EXCLUDED from that item's stats (they are
  reported separately as a footnote count, not folded into "dissatisfied").
- "Top-2-box" = share of respondents choosing one of the two most positive
  options (e.g., "Very satisfied" + "Somewhat satisfied").
- All percentages are of valid (non-missing, substantive) responses for
  that specific question — denominators therefore vary slightly by item
  and are reported alongside each stat.
- n = 1,749 total pre-program survey responses in the raw file.
"""

import json
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 0. Setup
# ----------------------------------------------------------------------
CSV_PATH = "Pre Program Survey Data-Career Launch.csv"
CHART_DIR = "charts"
import os
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
})

NAVY = "#1f3a5f"
GOLD = "#c9a227"
GREY = "#8a8f98"
PALETTE_5 = ["#a6192e", "#e07a5f", "#c9a227", "#7fa37f", "#1f6f54"]   # low -> high, 5-pt
PALETTE_3 = ["#a6192e", "#c9a227", "#1f6f54"]                        # low -> high, 3-pt

df = pd.read_csv(CSV_PATH)
N_TOTAL = len(df)

results = {"n_total_responses": N_TOTAL}

# ----------------------------------------------------------------------
# helper functions
# ----------------------------------------------------------------------
def clean_counts(series, order, drop_labels=None):
    """Return an ordered value_counts dict, dropping non-substantive labels."""
    s = series.dropna()
    drop_labels = drop_labels or []
    excluded_n = s.isin(drop_labels).sum()
    s = s[~s.isin(drop_labels)]
    counts = s.value_counts()
    counts = counts.reindex(order).fillna(0).astype(int)
    return counts, excluded_n, s.shape[0]


def pct_dict(counts, valid_n):
    return {k: round(100 * v / valid_n, 1) if valid_n else 0 for k, v in counts.items()}


def top2box(counts, valid_n, top_labels):
    top_n = sum(counts.get(l, 0) for l in top_labels)
    return round(100 * top_n / valid_n, 1) if valid_n else 0


def horizontal_bar_chart(labels, pct_values, colors, title, filename,
                          xlabel="% of respondents", figsize=(7.5, 3.6)):
    fig, ax = plt.subplots(figsize=figsize)
    y_pos = range(len(labels))
    bars = ax.barh(y_pos, pct_values, color=colors, edgecolor="white", height=0.62)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, max(pct_values) * 1.18 + 2)
    ax.set_title(title, fontsize=12, fontweight="bold", color=NAVY, pad=10)
    for bar, val in zip(bars, pct_values):
        ax.text(bar.get_width() + max(pct_values) * 0.015, bar.get_y() + bar.get_height() / 2,
                f"{val:.0f}%", va="center", fontsize=9.5, color="#222222")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{CHART_DIR}/{filename}", dpi=200)
    plt.close(fig)


def vertical_bar_chart(labels, values, colors, title, filename, ylabel,
                        figsize=(7.5, 4.2), value_fmt="{:.1f}"):
    fig, ax = plt.subplots(figsize=figsize)
    x_pos = range(len(labels))
    bars = ax.bar(x_pos, values, color=colors, edgecolor="white", width=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=18, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight="bold", color=NAVY, pad=10)
    ax.set_ylim(0, max(values) * 1.2)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values) * 0.02,
                value_fmt.format(val), ha="center", fontsize=9.5, color="#222222")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{CHART_DIR}/{filename}", dpi=200)
    plt.close(fig)


# ========================================================================
# 1. SATISFACTION WITH INTERNSHIP MATCH  (headline metric)
# ========================================================================
order5_sat = ["Very dissatisfied", "Somewhat dissatisfied", "Neutral",
              "Somewhat satisfied", "Very satisfied"]
counts, excluded, valid_n = clean_counts(df["How satisfied are you with your internship match?"],
                                          order5_sat)
pcts = pct_dict(counts, valid_n)
mean_rating = df["Satisfaction Rating"].mean()  # already 0-100 scale, aligned to same labels
top2 = top2box(counts, valid_n, ["Very satisfied", "Somewhat satisfied"])
bottom2 = top2box(counts, valid_n, ["Very dissatisfied", "Somewhat dissatisfied"])

results["satisfaction_with_match"] = {
    "valid_n": int(valid_n),
    "excluded_blank_n": int(excluded),
    "mean_score_0_100": round(mean_rating, 1),
    "pct_satisfied_top2box": top2,
    "pct_dissatisfied_bottom2box": bottom2,
    "distribution_pct": pcts,
    "distribution_counts": {k: int(v) for k, v in counts.items()},
}

horizontal_bar_chart(
    order5_sat, [pcts[l] for l in order5_sat], PALETTE_5,
    "Satisfaction with Internship Match\n(pre-program survey)",
    "01_satisfaction_with_match.png",
)

# ========================================================================
# 2. SATISFACTION WITH THE PLACEMENT MATCHING PROCESS OVERALL
# ========================================================================
counts2, excluded2, valid_n2 = clean_counts(
    df["How satisfied are you with the placement matching process overall?"], order5_sat)
pcts2 = pct_dict(counts2, valid_n2)
top2_process = top2box(counts2, valid_n2, ["Very satisfied", "Somewhat satisfied"])

results["satisfaction_with_process_overall"] = {
    "valid_n": int(valid_n2),
    "excluded_blank_n": int(excluded2),
    "pct_satisfied_top2box": top2_process,
    "distribution_pct": pcts2,
    "distribution_counts": {k: int(v) for k, v in counts2.items()},
}

horizontal_bar_chart(
    order5_sat, [pcts2[l] for l in order5_sat], PALETTE_5,
    "Satisfaction with the Placement Matching\nProcess Overall",
    "02_satisfaction_with_process.png",
)

# ========================================================================
# 3. MATCH vs. OVERALL PROCESS — side-by-side top-2-box comparison
# ========================================================================
vertical_bar_chart(
    ["Satisfied with\ntheir own match", "Satisfied with the\nmatching process overall"],
    [top2, top2_process], [NAVY, GOLD],
    "Share of Students Satisfied (Top-2-Box)",
    "03_match_vs_process_top2box.png",
    ylabel="% satisfied (Very + Somewhat)",
    value_fmt="{:.0f}%",
)

# ========================================================================
# 4. HUB SUPPORT & PREP: helpfulness / clarity items
# ========================================================================
order5_agree = ["Strongly Disagree", "Somewhat Disagree", "Neither Agree nor Disagree",
                "Somewhat Agree", "Strongly Agree"]
counts_support, excl_support, n_support = clean_counts(
    df["I feel supported by my hub staff as I prepare to start my internship."], order5_agree)
pcts_support = pct_dict(counts_support, n_support)
top2_support = top2box(counts_support, n_support, ["Strongly Agree", "Somewhat Agree"])

order3_help = ["Not helpful", "Somewhat helpful", "Very helpful"]
non_substantive_help = ["I did not attend", "I don't remember", "I don’t remember"]

counts_workshop, excl_workshop, n_workshop = clean_counts(
    df["How helpful was the workshop on InPlace & Career Eco in preparing you to apply and interview for placements?"],
    order3_help, drop_labels=non_substantive_help)
counts_tips, excl_tips, n_tips = clean_counts(
    df["How helpful were the interview tips and resume prep provided by your hub?"],
    order3_help, drop_labels=non_substantive_help)

order3_clear = ["Not clear", "Somewhat clear", "Very clear"]
counts_clear, excl_clear, n_clear = clean_counts(
    df["How clear were the instructions about how to apply and interview for placements?"],
    order3_clear, drop_labels=["I did not receive any instructions"])

pcts_workshop = pct_dict(counts_workshop, n_workshop)
pcts_tips = pct_dict(counts_tips, n_tips)
pcts_clear = pct_dict(counts_clear, n_clear)

results["hub_support_and_prep"] = {
    "feel_supported_by_hub": {
        "valid_n": int(n_support), "pct_agree_top2box": top2_support,
        "distribution_pct": pcts_support,
    },
    "workshop_helpfulness": {
        "valid_n": int(n_workshop), "excluded_non_substantive_n": int(excl_workshop),
        "pct_helpful_top2box": top2box(counts_workshop, n_workshop, ["Very helpful", "Somewhat helpful"]),
        "distribution_pct": pcts_workshop,
    },
    "interview_tips_helpfulness": {
        "valid_n": int(n_tips), "excluded_non_substantive_n": int(excl_tips),
        "pct_helpful_top2box": top2box(counts_tips, n_tips, ["Very helpful", "Somewhat helpful"]),
        "distribution_pct": pcts_tips,
    },
    "instructions_clarity": {
        "valid_n": int(n_clear), "excluded_non_substantive_n": int(excl_clear),
        "pct_clear_top2box": top2box(counts_clear, n_clear, ["Very clear", "Somewhat clear"]),
        "distribution_pct": pcts_clear,
    },
}

# grouped chart: 3 prep items, top-2-box %
prep_labels = ["Felt supported\nby hub staff", "InPlace/Career Eco\nworkshop helpful",
               "Interview tips &\nresume prep helpful", "Application/interview\ninstructions clear"]
prep_values = [top2_support,
               results["hub_support_and_prep"]["workshop_helpfulness"]["pct_helpful_top2box"],
               results["hub_support_and_prep"]["interview_tips_helpfulness"]["pct_helpful_top2box"],
               results["hub_support_and_prep"]["instructions_clarity"]["pct_clear_top2box"]]
vertical_bar_chart(
    prep_labels, prep_values, [NAVY, GOLD, "#7fa37f", "#1f6f54"],
    "Hub Prep & Support — % Positive Response",
    "04_hub_prep_support.png",
    ylabel="% positive (top-2-box)",
    value_fmt="{:.0f}%",
    figsize=(8, 4.4),
)

# ========================================================================
# 5. WORKPLACE READINESS SKILLS (1-5 self-rated confidence, 6 items)
# ========================================================================
skill_cols = [
    "Professional Communication (email, meetings)",
    "TIme management at work",
    "Asking for help or clarification",
    "Using workplace technology/tools",
    "Understanding workplace expectations",
    "Working with supervisors",
]
skill_labels = ["Professional\ncommunication", "Time\nmanagement", "Asking for help/\nclarification",
                "Workplace tech\n/tools", "Understanding\nexpectations", "Working with\nsupervisors"]
skill_means = [df[c].mean() for c in skill_cols]

results["workplace_readiness_skills_1to5"] = {
    label.replace("\n", " "): round(m, 2) for label, m in zip(skill_labels, skill_means)
}
results["workplace_readiness_skills_1to5"]["valid_n"] = int(N_TOTAL)

vertical_bar_chart(
    skill_labels, skill_means, [NAVY] * len(skill_labels),
    "Self-Rated Workplace Readiness\n(1 = not confident, 5 = very confident)",
    "05_workplace_readiness_skills.png",
    ylabel="Mean rating (1-5)",
    figsize=(8.2, 4.4),
)

# ========================================================================
# 6. CAREER OUTLOOK — belief internship helps + confidence items
# ========================================================================
counts_help_job, _, n_help_job = clean_counts(
    df["I believe that my internship will help me get a job in the future"], ["No", "Unsure", "Yes"])
pct_help_job = pct_dict(counts_help_job, n_help_job)

order5_agree2 = ["Strongly Disagree", "Somewhat Disagree", "Neutral", "Somewhat Agree", "Strongly Agree"]
outlook_cols = {
    "I feel prepared to search for and apply to jobs in my field": "Feel prepared to\nsearch/apply for jobs",
    "I am optimistic about my ability to find a job in my field within the next year": "Optimistic about finding\na job within a year",
    "I feel confident in my ability to perform well in a job interview": "Confident performing\nin a job interview",
}
outlook_top2 = {}
for col, label in outlook_cols.items():
    c, _, n = clean_counts(df[col], order5_agree2)
    outlook_top2[label] = top2box(c, n, ["Strongly Agree", "Somewhat Agree"])

results["career_outlook"] = {
    "believes_internship_helps_get_job_pct": pct_help_job,
    "pct_agree_top2box_by_item": outlook_top2,
}

vertical_bar_chart(
    list(outlook_top2.keys()), list(outlook_top2.values()),
    [NAVY, GOLD, "#7fa37f"],
    "Career Outlook — % Agreeing (Top-2-Box)",
    "06_career_outlook.png",
    ylabel="% agree (Strongly + Somewhat)",
    value_fmt="{:.0f}%",
    figsize=(8, 4.4),
)

# ========================================================================
# 7. PRIOR INTERNSHIP APPLICATION EXPERIENCE
# ========================================================================
prior_apps = df["Before Career Launch, how many times did you apply for an internship or CUNY internship program (regardless of whether you were accepted)?"]
n_zero = int((prior_apps == 0).sum())
n_valid_apps = int(prior_apps.notna().sum())
pct_zero_prior = round(100 * n_zero / n_valid_apps, 1)
median_apps = float(prior_apps.median())
mean_apps = float(prior_apps.mean())

results["prior_application_experience"] = {
    "valid_n": n_valid_apps,
    "pct_with_zero_prior_applications": pct_zero_prior,
    "median_prior_applications": median_apps,
    "mean_prior_applications": round(mean_apps, 2),
}

# bucket for a clean chart
def bucket_apps(x):
    if x == 0:
        return "0"
    if x == 1:
        return "1"
    if x <= 3:
        return "2-3"
    if x <= 9:
        return "4-9"
    return "10+"

buckets_order = ["0", "1", "2-3", "4-9", "10+"]
bucket_counts = prior_apps.dropna().apply(bucket_apps).value_counts().reindex(buckets_order).fillna(0)
bucket_pcts = [round(100 * v / n_valid_apps, 1) for v in bucket_counts]

vertical_bar_chart(
    buckets_order, bucket_pcts, [GREY, "#c9a227", "#e07a5f", NAVY, "#1f6f54"],
    "Prior Internship Applications\n(before Career Launch)",
    "07_prior_applications.png",
    ylabel="% of respondents",
    value_fmt="{:.0f}%",
)

# ========================================================================
# 8. SEEKING A PERMANENT ROLE AFTER THE INTERNSHIP
# ========================================================================
perm_counts, _, perm_n = clean_counts(
    df["Are you seeking a permanent role at your internship site or a similar site after you complete your internship? "],
    ["No", "Unsure", "Yes"])
perm_pcts = pct_dict(perm_counts, perm_n)
results["seeking_permanent_role"] = {"valid_n": int(perm_n), "distribution_pct": perm_pcts}

horizontal_bar_chart(
    ["No", "Unsure", "Yes"], [perm_pcts["No"], perm_pcts["Unsure"], perm_pcts["Yes"]],
    ["#a6192e", "#c9a227", "#1f6f54"],
    "Seeking a Permanent Role at Internship Site\n(or similar) After the Program",
    "08_seeking_permanent_role.png",
    figsize=(7.5, 2.8),
)

# ========================================================================
# Save summary
# ========================================================================
with open("summary_tables.json", "w") as f:
    json.dump(results, f, indent=2)

print("Done.")
print(f"Total responses: {N_TOTAL}")
print(f"Satisfied with match (top-2-box): {top2}% (n={valid_n})")
print(f"Satisfied with process overall (top-2-box): {top2_process}% (n={valid_n2})")
print("Charts saved to ./charts/, summary saved to summary_tables.json")

Done.
Total responses: 1749
Satisfied with match (top-2-box): 79.0% (n=1680)
Satisfied with process overall (top-2-box): 76.0% (n=1680)
Charts saved to ./charts/, summary saved to summary_tables.json
